# Environment Setup

In [10]:
!pip -q install datasets==3.6.0
!pip -q install -U huggingface_hub hf_transfer
!pip -q install -U duckdb huggingface_hub

In [11]:
# Importing all the dependencies
from datasets import load_dataset
import pandas as pd
import numpy as np
from typing import Tuple, Optional, Union
import os, time, duckdb
from huggingface_hub import login, HfApi, list_repo_files
import gdown
from google.colab import userdata

In [3]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Retrieve the token from Colab's Secrets Manager
hf_token = userdata.get('HF_TOKEN')

# Log in to HF using the retrieved token
login(hf_token)

# Working with Hugging Face Datasets

### Downloading files using HF datasets

In [3]:
# Dwnloading the data from hugging face datasets.
reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Clothing_Shoes_and_Jewelry", trust_remote_code=True)
items = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Clothing_Shoes_and_Jewelry", split="full", trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading dataset shards:   0%|          | 0/38 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

In [116]:
print(reviews["full"])
print(items["full"])

Dataset({
    features: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase'],
    num_rows: 66033346
})

### Counting the Non-Null values in each column.

In [100]:
def count_non_null(batch, columns):
    """Count non-null values in specified columns for a batch"""
    counts = {col: [] for col in columns}
    batch_size = len(batch[list(batch.keys())[0]]) # Get the size of the batch

    for i in range(batch_size):
        for col in columns:
            if col in batch:
                # Check if the value for the current example and column is not None
                # IN CASE OF PRICE COLUMN IT IS 'None' STRING and in case of features and descriptions it is '[]' empty list,
                # so do the appropriate changes accordingly in below code
                counts[col].append(1 if batch[col][i] is not None else 0)
            else:
                # If column not in batch, append 0 for this example
                counts[col].append(0)
    return counts

# Columns to check
columns = ['average_rating', 'rating_number']

# Count non-null values with multiprocessing
results = items.map(
    count_non_null,
    fn_kwargs={'columns': columns},
    batched=True,
    batch_size=1000,
    num_proc=4,  # Use 4 processes
    remove_columns=items.column_names  # Remove original columns to save memory
)

# Aggregate results
final_counts = {col: sum(results[col]) for col in columns}

# Print results
print("Non-null value counts:")
for col, count in final_counts.items():
    print(f"- {col}: {count:,} ({(count/len(items))*100:.1f}%)")

Map (num_proc=4):   0%|          | 0/7218481 [00:00<?, ? examples/s]

Non-null value counts:
- average_rating: 7,218,481.0 (100.0%)
- rating_number: 7,218,481 (100.0%)


In [106]:
# NOT RECOMMENDED: This is a very bad way to check null values as the RAM will spike a lot.
column_name = 'helpful_vote'
x =reviews['full'][column_name]
print(x[:10])
add_ = 0
for i in x:
  if i == 0:
    add_ += 1
print(add_)

### Benchmarking the performance of HF datasets and DuckDB

In [42]:
# To benchmark the difference in execution time of HF datasets and DuckDB!!!!
NPROC = min(8, os.cpu_count() or 2)

t0 = time.perf_counter()
# Filter keeps only verified rows (runs in parallel on CPU)
verified_ds = reviews.filter(lambda x: bool(x["verified_purchase"]), num_proc=NPROC)
count_ds = verified_ds.num_rows['full']
t1 = time.perf_counter()

print(f"[datasets] verified count = {count_ds:,}  | time = {t1 - t0:.2f}s  | num_proc={NPROC}")

In [6]:
# TO use DuckDB we need to first convert the required data to parquet format for maximum efficiency.
parquet_dir = "/content/reviews_parquet"

# Keep only what we need for this quick benchmark to keep files tiny
need_cols = [c for c in reviews['full'].column_names if c in ("verified_purchase",)]
reviews_small = reviews['full'].remove_columns([c for c in reviews['full'].column_names if c not in need_cols])

# Write shards to Parquet (this creates multiple files under the folder)
reviews_small.to_parquet(parquet_dir)

In [39]:
parquet_path = "/content/reviews_parquet"  # <-- use YOUR actual file path

con = duckdb.connect()
con.execute(f"PRAGMA threads={min(8, os.cpu_count() or 2)};")
con.execute("PRAGMA memory_limit='8GB';")

t0 = time.perf_counter()
count_duck = con.execute("""
  SELECT COUNT(*)
  FROM read_parquet(?)
  WHERE verified_purchase = TRUE
""", [parquet_path]).fetchone()[0]
t1 = time.perf_counter()

print(f"[duckdb ] verified count = {count_duck:,} | time = {t1 - t0:.2f}s")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[duckdb ] verified count = 62,175,766 | time = 2.16s


***CLEARLY DUCKDB IS BLAZINGLY FAST !!!***

### Preprocessing before converting to parquet.

In [113]:
# REVIEWS — keep slim columns, filter verified
rev_keep = ["user_id","parent_asin","timestamp","rating","verified_purchase","helpful_vote"]
reviews_slim = reviews["full"].remove_columns([c for c in reviews["full"].column_names if c not in rev_keep])
reviews_slim = reviews_slim.filter(lambda x: bool(x["verified_purchase"]), num_proc=4)
# (Optional) drop the flag now that you’ve filtered:
reviews_slim = reviews_slim.remove_columns(["verified_purchase"])

Filter (num_proc=4):   0%|          | 0/66033346 [00:00<?, ? examples/s]

In [125]:
# ITEMS — extract only what you need; we’ll map images→main_image_url later
itm_keep = ["parent_asin","main_category","title","average_rating","rating_number","price","images","categories","features","description","categories","details"]
items_slim = items.remove_columns([c for c in items.column_names if c not in itm_keep])

### Extracting the Main Image URL

In [127]:
NPROC = min(8, os.cpu_count() or 2)

def _first_url(val):
    """Return the first non-empty string URL found inside val (str/list/ndarray/dict)."""
    if val is None:
        return None
    if isinstance(val, str):
        s = val.strip()
        return s or None
    if isinstance(val, (list, tuple, np.ndarray)):
        for x in val:
            u = _first_url(x)
            if u:
                return u
        return None
    if isinstance(val, dict):
        # common keys in the Amazon dumps; prioritize hi_res -> large -> medium -> url
        for k in ("hi_res", "large", "thumb"):
            if k in val:
                u = _first_url(val[k])
                if u:
                    return u
        return None
    # anything else
    return None

def to_main_url(batch):
    """datasets.map(batched=True) callback: reads batch['images'] (dicts) -> main_image_url"""
    out = []
    for img in batch["images"]:
        url = None
        if isinstance(img, dict):
            url = _first_url(img)           # dict case (your schema)
        else:
            url = _first_url(img)           # be tolerant to accidental list/str formats
        out.append(url)
    batch["main_image_url"] = out
    return batch


In [128]:
# items_slim must contain the 'images' column
items_slim = items_slim.map(to_main_url, batched=True, num_proc=NPROC)
items_slim = items_slim.remove_columns(["images"])

Map (num_proc=8):   0%|          | 0/7218481 [00:00<?, ? examples/s]

In [133]:
# Checking how many products there are without ant image.
len(items_slim.filter(lambda x: x["main_image_url"] is None, num_proc=NPROC))

Filter (num_proc=8):   0%|          | 0/7218481 [00:00<?, ? examples/s]

10048

### Converting to parquet format

In [136]:
rev_path = "/content/reviews_small_unpart"
itm_path = "/content/items_small_unpart"
reviews_slim.to_parquet(rev_path)

size_bytes = os.path.getsize(rev_path)

def human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024:
            return f"{n:.2f} {u}"
        n /= 1024
print("size:", human(size_bytes))

items_slim.to_parquet(itm_path)

Creating parquet from Arrow format:   0%|          | 0/62176 [00:00<?, ?ba/s]

4352307484

# Downloading the parquet file from Google Drive Link

In [4]:
# Your shared links (file IDs extracted below)
REV_ID = "1xTxaWEYpCxIVJPwprG4NFUL02SyRWQsn"
ITM_ID = "155CP80vSsf2CNp3DUlXQ_A9EgMAvL_Wq"

REV_OUT = "/content/reviews_small_unpart"
ITM_OUT = "/content/items_small_unpart"

gdown.download(id=REV_ID, output=REV_OUT, quiet=False)
gdown.download(id=ITM_ID, output=ITM_OUT, quiet=False)

# (optional) show sizes
def human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.2f} {u}"
        n /= 1024
    return f"{n:.2f} PB"

print("reviews size:", human(os.path.getsize(REV_OUT)))
print("items   size:", human(os.path.getsize(ITM_OUT)))

Downloading...
From (original): https://drive.google.com/uc?id=1xTxaWEYpCxIVJPwprG4NFUL02SyRWQsn
From (redirected): https://drive.google.com/uc?id=1xTxaWEYpCxIVJPwprG4NFUL02SyRWQsn&confirm=t&uuid=3b8a4dd0-e072-4bdf-97c1-8e80438c7f03
To: /content/reviews_small_unpart
100%|██████████| 1.91G/1.91G [00:37<00:00, 51.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=155CP80vSsf2CNp3DUlXQ_A9EgMAvL_Wq
From (redirected): https://drive.google.com/uc?id=155CP80vSsf2CNp3DUlXQ_A9EgMAvL_Wq&confirm=t&uuid=010a63ec-997a-40ec-997a-9a3d7d7aa91f
To: /content/items_small_unpart
100%|██████████| 4.20G/4.20G [00:53<00:00, 78.8MB/s]

reviews size: 1.78 GB
items   size: 3.91 GB


In [5]:
st = time.time()
# Load the parquet files into datasets
reviews = load_dataset("parquet", data_files=REV_OUT)
items = load_dataset("parquet", data_files=ITM_OUT)

et = time.time()
print("Total execution time:", et - st, "seconds")
# You might want to inspect the loaded datasets
print("Reviews dataset:")
print(reviews)
print("\nItems dataset:")
print(items)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

Total execution time: 176.02103090286255 seconds
Reviews dataset:
DatasetDict({
    train: Dataset({
        features: ['rating', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote'],
        num_rows: 62175766
    })
})

Items dataset:
DatasetDict({
    train: Dataset({
        features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'categories', 'details', 'parent_asin', 'main_image_url'],
        num_rows: 7218481
    })
})


# Loading and Uploading data using HF

In [6]:
api = HfApi()

repo_id = "PirateKing0402/Amazon_dataset"   # change this
api.create_repo(repo_id=repo_id, repo_type="dataset", private=False, exist_ok=True)

RepoUrl('https://huggingface.co/datasets/PirateKing0402/Amazon_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='PirateKing0402/Amazon_dataset')

### Uploading data to HF

In [12]:
api.upload_file(
    path_or_fileobj=REV_OUT,
    path_in_repo="reviews_small_unpart",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Add reviews parquet unpartitioned (hf_transfer)"
)

api.upload_file(
    path_or_fileobj=ITM_OUT,
    path_in_repo="items_small_unpart",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Add items parquet unpartitioned (hf_transfer)"
)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /content/reviews_small_unpart         :   0%|          |  544kB / 1.91GB            

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /content/items_small_unpart           :   0%|          |  525kB / 4.20GB            

CommitInfo(commit_url='https://huggingface.co/datasets/PirateKing0402/Amazon_dataset/commit/a60281c9ad533dffaf04fc91a0b72f7ec7c29480', commit_message='Add items parquet unpartitioned (hf_transfer)', commit_description='', oid='a60281c9ad533dffaf04fc91a0b72f7ec7c29480', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/PirateKing0402/Amazon_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='PirateKing0402/Amazon_dataset'), pr_revision=None, pr_num=None)

### Loading data from HF
( Slower than Google Drive download, better to use DuckDB to get data from HF )

In [7]:
# Remove the reviews and items variables from memory
del reviews
del items

# You can optionally add print statements to confirm they are deleted (will raise NameError if successful)
# print(reviews)
# print(items)

In [9]:
from datasets import load_dataset

# Define the repository ID and file paths within the repo
repo_id = "PirateKing0402/Amazon_dataset"
reviews_file_path = "reviews_small_unpart"
items_file_path = "items_small_unpart"
st = time.time()
# Load the parquet files into datasets
reviews = load_dataset("parquet", data_files=f"hf://datasets/{repo_id}/{reviews_file_path}")
items = load_dataset("parquet", data_files=f"hf://datasets/{repo_id}/{items_file_path}")
et = time.time()

print("Total execution time:", et - st, "seconds")
# You might want to inspect the loaded datasets
print("Reviews dataset:")
print(reviews)
print("\nItems dataset:")
print(items)

reviews_small_unpart:   0%|          | 0.00/1.91G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

items_small_unpart:   0%|          | 0.00/4.20G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

Total execution time: 969.474189043045 seconds
Reviews dataset:
DatasetDict({
    train: Dataset({
        features: ['rating', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote'],
        num_rows: 62175766
    })
})

Items dataset:
DatasetDict({
    train: Dataset({
        features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'categories', 'details', 'parent_asin', 'main_image_url'],
        num_rows: 7218481
    })
})


# Processing using DuckDB

### Using DuckDB on a data accessed through Remote Connection.
DuckDB's power lies in its ability to query massive remote datasets efficiently without local downloads. It achieves this by making many small, precise HTTP range requests to read only the parts of a file it needs.

This method is perfectly suited for cloud object stores like Amazon S3 or GCS, which are designed for this high-throughput access pattern. However, it triggers the anti-bot rate limits on standard web servers like the Hugging Face Hub, causing the query to fail.

In [16]:
# ---- config: your repo + paths (file OR folder)
REPO_ID = "PirateKing0402/Amazon_dataset"
REV_PREFIX = "reviews_small_unpart"  # e.g. "reviews_small_unpart.parquet" OR a folder "reviews_small_unpart/"
ITM_PREFIX = "items_small_unpart"    # same idea for items

# Helper: build HTTPS URLs to the exact parquet files in the repo
def hf_parquet_urls(repo_id: str, prefix: str):
    files = list_repo_files(repo_id, repo_type="dataset")
    # case 1: a single file like "<prefix>.parquet"
    exact = [p for p in files if p == f"{prefix}"]
    if exact:
        return [f"https://huggingface.co/datasets/{repo_id}/resolve/main/{exact[0]}"]

rev_urls = hf_parquet_urls(REPO_ID, REV_PREFIX)
itm_urls = hf_parquet_urls(REPO_ID, ITM_PREFIX)

# Safety check: make sure we found files
print("review files:", len(rev_urls))
print("item files  :", len(itm_urls))
assert rev_urls, "No review parquet found in the repo/path you provided."
assert itm_urls, "No item parquet found in the repo/path you provided."

review files: 1
item files  : 1


In [25]:
# ---- DuckDB setup (HTTP range reads; only needed bytes fetched)
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='8GB';")  # optional

# ---- Items: count duplicate rows by parent_asin
sql_items = """
WITH g AS (
  SELECT parent_asin, COUNT(*) AS cnt
  FROM read_parquet($urls)
  GROUP BY parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,        -- matches pandas .duplicated(...).sum()
  COUNT(*) AS keys_with_duplicates  -- number of parent_asin groups that had dupes
FROM g;
"""
items_dup = con.execute(sql_items, {"urls": itm_urls}).fetchdf()

In [ ]:
sql_reviews = """
WITH g AS (
  SELECT user_id, parent_asin, COUNT(*) AS cnt
  FROM read_parquet($urls)
  GROUP BY user_id, parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,            -- matches pandas .duplicated(...).sum()
  COUNT(*)                    AS keys_with_duplicates      -- number of (user,item) pairs that had dupes
FROM g;
"""
reviews_dup = con.execute(sql_reviews, {"urls": rev_urls}).fetchdf()

print("\nItems duplicates (pandas equivalence): duplicated(subset=['parent_asin']).sum()")
print(int(items_dup.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(items_dup.loc[0, "keys_with_duplicates"]))

print("\nReviews duplicates (pandas equivalence): duplicated(subset=['user_id','parent_asin']).sum()")
print(int(reviews_dup.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(reviews_dup.loc[0, "keys_with_duplicates"]))

### Using DuckDB on locally downloaded data.

In [26]:
# ---- DuckDB setup (using local files)
con = duckdb.connect()
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='8GB';")  # optional

# Define local file paths
REV_PATH_LOCAL = "/content/reviews_small_unpart"
ITM_PATH_LOCAL = "/content/items_small_unpart"

# ---- Items: count duplicate rows by parent_asin using local file
sql_items_local = """
WITH g AS (
  SELECT parent_asin, COUNT(*) AS cnt
  FROM read_parquet(?)
  GROUP BY parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,        -- matches pandas .duplicated(...).sum()
  COUNT(*) AS keys_with_duplicates  -- number of parent_asin groups that had dupes
FROM g;
"""
items_dup_local = con.execute(sql_items_local, [ITM_PATH_LOCAL]).fetchdf()

print("Items duplicates (local file):")
print(int(items_dup_local.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(items_dup_local.loc[0, "keys_with_duplicates"]))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Items duplicates (local file):
0 | keys_with_duplicates: 0


In [27]:
# ---- Reviews: count duplicate rows by user_id and parent_asin using local file
sql_reviews_local = """
WITH g AS (
  SELECT user_id, parent_asin, COUNT(*) AS cnt
  FROM read_parquet(?)
  GROUP BY user_id, parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,            -- matches pandas .duplicated(...).sum()
  COUNT(*)                    AS keys_with_duplicates      -- number of (user,item) pairs that had dupes
FROM g;
"""
reviews_dup_local = con.execute(sql_reviews_local, [REV_PATH_LOCAL]).fetchdf()

print("\nReviews duplicates (local file):")
print(int(reviews_dup_local.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(reviews_dup_local.loc[0, "keys_with_duplicates"]))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Reviews duplicates (local file):
803670 | keys_with_duplicates: 684012


# Rest

In [142]:
######### CHECK FOR DUPLICATESS #################
print(main_items_df.duplicated(subset=["parent_asin"]).sum())
print(main_reviews_df.duplicated(subset=["user_id","parent_asin"]).sum())

NameError: name 'main_items_df' is not defined

In [ ]:
ts_num=pd.to_numeric(main_reviews_df["timestamp"], errors="coerce")
pd.to_datetime(ts_num, unit="ms", utc=True)

,timestamp
0,2020-01-09 00:06:34.489000+00:00
1,2020-12-20 01:04:06.701000+00:00
2,2015-05-23 01:33:48+00:00
3,2018-12-31 20:57:27.095000+00:00
4,2015-08-13 14:29:26+00:00
...,...
2500934,2016-06-24 20:12:38+00:00
2500935,2018-05-08 17:05:05.585000+00:00
2500936,2016-12-17 22:28:31+00:00
2500937,2017-04-15 17:34:26+00:00


In [ ]:
TimestampLike = Union[pd.Timestamp, str, int, float]

def temporal_split_ms(
    df: pd.DataFrame,
    time_col: str,
    *,
    test_fraction: Optional[float] = None,   # exactly one of these
    cutoff: Optional[TimestampLike] = None,  # not both
    train_includes_cutoff: bool = True,
    drop_na_time: bool = True,
    sort_within_splits: bool = False,        # sort by the ms column itself
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Timestamp]:
    # ---- input checks
    if (test_fraction is None) == (cutoff is None):
        raise ValueError("Provide exactly one of `test_fraction` or `cutoff`.")
    if test_fraction is not None:
        if not np.isfinite(test_fraction):
            raise ValueError("`test_fraction` must be finite.")
        if not (0.0 < float(test_fraction) < 1.0):
            raise ValueError("`test_fraction` must be in (0, 1).")

    # ---- ensure numeric ms epoch
    ms = pd.to_numeric(df[time_col], errors="coerce")

    # drop invalid timestamps if requested
    if drop_na_time:
        valid = ms.notna()
        if not valid.all():
            df = df.loc[valid]
            ms = ms.loc[valid]
    if len(df) == 0:
        raise ValueError("All timestamps are NaN after parsing; nothing to split.")

    # ---- pick cutoff in **ms**
    if cutoff is None:
        q = 1.0 - float(test_fraction)
        cutoff_ms = ms.quantile(q, interpolation="nearest")
        if not np.isfinite(cutoff_ms):
            raise RuntimeError("Failed to compute a valid quantile cutoff.")
        cutoff_ms = int(cutoff_ms)
    else:
        if isinstance(cutoff, (int, float)) and np.isfinite(cutoff):
            cutoff_ms = int(cutoff)
        else:
            # parse to UTC and convert to ms
            ts = pd.to_datetime(cutoff, utc=True)
            if pd.isna(ts):
                raise ValueError("`cutoff` could not be parsed into a valid timestamp.")
            cutoff_ms = int(ts.value // 1_000_000)  # ns -> ms

    # sanity: cutoff within range
    mn, mx = ms.min(), ms.max()
    if not (mn <= cutoff_ms <= mx):
        raise RuntimeError(f"Cutoff {cutoff_ms} outside data range [{mn}, {mx}].")

    # ---- split using numeric masks (no datetime needed)
    if train_includes_cutoff:
        train_mask = ms.le(cutoff_ms)
        test_mask  = ms.gt(cutoff_ms)
    else:
        train_mask = ms.lt(cutoff_ms)
        test_mask  = ms.ge(cutoff_ms)

    train = df.loc[train_mask].copy()
    test  = df.loc[test_mask].copy()

    if len(train) == 0 or len(test) == 0:
        raise RuntimeError(
            f"Empty split: train={len(train)}, test={len(test)}. "
            "Adjust `test_fraction` or `cutoff`."
        )
    if not train.index.intersection(test.index).empty:
        raise AssertionError("Split overlap detected (indices intersect).")

    # ---- optional: sort by the ms column itself (stable if you want ties preserved)
    if sort_within_splits:
        train = train.sort_values(time_col, kind="mergesort")
        test  = test.sort_values(time_col, kind="mergesort")

    # return cutoff as a human-readable Timestamp (naive UTC)
    cutoff_ts = pd.to_datetime(cutoff_ms, unit="ms", utc=True).tz_convert(None)
    return train, test, cutoff_ts


In [ ]:
# 80/20 temporal split
train_df, test_df, cutoff_ts = temporal_split_ms(
    main_reviews_df, time_col="timestamp", test_fraction=0.2, sort_within_splits=True
)
print("Cutoff:", cutoff_ts, "Train:", len(train_df), "Test:", len(test_df))


Cutoff: 2020-12-31 19:23:33.359000 Train: 1870162 Test: 467540


In [ ]:
def kcore_filter_iterative(
    df: pd.DataFrame,
    user_col: str = "user_id",
    item_col: str = "parent_asin",
    user_k: int = 5,
    item_k: int = 5,
    max_iters: int = 100,
    drop_duplicates: bool = False,        # drop exact duplicate (user,item,...) rows first
    return_history: bool = False,
    verbose: bool = False,
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Iteratively prune users/items with degree < thresholds until convergence.
    Degrees are counted as row frequency (after optional de-duplication).
    Returns (filtered_df, history_df or None).
    """

    # ---- input checks
    if user_col not in df.columns or item_col not in df.columns:
        raise ValueError(f"Missing required columns: {user_col!r}, {item_col!r}")
    if not (isinstance(user_k, int) and user_k >= 1):
        raise ValueError("user_k must be an integer >= 1.")
    if not (isinstance(item_k, int) and item_k >= 1):
        raise ValueError("item_k must be an integer >= 1.")
    if not (isinstance(max_iters, int) and max_iters >= 1):
        raise ValueError("max_iters must be an integer >= 1.")

    cur = df
    if drop_duplicates:
        cur = cur.drop_duplicates(subset=[user_col, item_col], keep="first")

    if len(cur) == 0:
        raise ValueError("No rows to process after de-duplication (if enabled).")

    history = []
    prev_len = -1

    for it in range(1, max_iters + 1):
        n_before = len(cur)

        # prune users
        ucnt = cur[user_col].value_counts()
        cur = cur[cur[user_col].map(ucnt) >= user_k]
        if len(cur) == 0:
            raise RuntimeError(f"All rows pruned at user step (iter={it}). "
                               f"Consider lowering user_k/item_k.")

        # prune items (recompute counts after user pruning)
        icnt = cur[item_col].value_counts()
        cur = cur[cur[item_col].map(icnt) >= item_k]
        if len(cur) == 0:
            raise RuntimeError(f"All rows pruned at item step (iter={it}). "
                               f"Consider lowering user_k/item_k.")

        n_after = len(cur)
        step = {
            "iter": it,
            "rows_before": n_before,
            "rows_after": n_after,
            "users_after": cur[user_col].nunique(),
            "items_after": cur[item_col].nunique(),
            "removed": n_before - n_after,
        }
        history.append(step)
        if verbose:
            print(step)

        # convergence: no change
        if n_after == n_before or n_after == prev_len:
            break
        prev_len = n_after

    hist_df = pd.DataFrame(history)
    return (cur.reset_index(drop=True), hist_df if return_history else None)


In [ ]:
# Example on train_df from your temporal split
warm_train_df, hist = kcore_filter_iterative(
    train_df,
    user_col="user_id",
    item_col="parent_asin",
    user_k=4,
    item_k=2,
    return_history=True,
    verbose=True
)

{'iter': 1, 'rows_before': 1850504, 'rows_after': 37343, 'users_after': 16159, 'items_after': 11746, 'removed': 1813161}
{'iter': 2, 'rows_before': 37343, 'rows_after': 8088, 'users_after': 2425, 'items_after': 2348, 'removed': 29255}
{'iter': 3, 'rows_before': 8088, 'rows_after': 3995, 'users_after': 939, 'items_after': 992, 'removed': 4093}
{'iter': 4, 'rows_before': 3995, 'rows_after': 2954, 'users_after': 605, 'items_after': 679, 'removed': 1041}
{'iter': 5, 'rows_before': 2954, 'rows_after': 2585, 'users_after': 503, 'items_after': 568, 'removed': 369}
{'iter': 6, 'rows_before': 2585, 'rows_after': 2419, 'users_after': 460, 'items_after': 525, 'removed': 166}
{'iter': 7, 'rows_before': 2419, 'rows_after': 2351, 'users_after': 443, 'items_after': 508, 'removed': 68}
{'iter': 8, 'rows_before': 2351, 'rows_after': 2329, 'users_after': 437, 'items_after': 504, 'removed': 22}
{'iter': 9, 'rows_before': 2329, 'rows_after': 2316, 'users_after': 434, 'items_after': 500, 'removed': 13}
{'i

In [ ]:
merged_train_df = pd.merge(warm_train_df, main_items_df, on='parent_asin', how='left')

In [ ]:
merged_train_df.shape

(2302, 25)

In [ ]:
merged_train_df.duplicated(subset=['user_id', 'parent_asin']).sum()

np.int64(0)

In [ ]:
merged_train_df["timestamp"]

,timestamp
0,1347420778000
1,1351210541000
2,1351210649000
3,1351212742000
4,1354641921000
...,...
2297,1579584725798
2298,1581085900142
2299,1585692925624
2300,1599164172226


In [ ]:
merged_train_df["main_image"]

,main_image
0,https://m.media-amazon.com/images/I/71e7WkcRQp...
1,https://m.media-amazon.com/images/I/51YVRMMK7L...
2,https://m.media-amazon.com/images/I/71lYjMM16S...
3,None
4,https://m.media-amazon.com/images/I/71lYjMM16S...
...,...
2297,https://m.media-amazon.com/images/I/611lH6Kj7+...
2298,https://m.media-amazon.com/images/I/71GFOKuvbh...
2299,https://m.media-amazon.com/images/I/61OTapGJEr...
2300,https://m.media-amazon.com/images/I/61kiAzBuBI...


In [ ]:
# Now, drop only the rows where no valid image could be found at all
cleaned_df = merged_train_df.dropna(subset=['main_image'])

In [ ]:
cleaned_df.shape

(2120, 26)